<a href="https://colab.research.google.com/github/stefkong1982/netology.ru/blob/Master/DS_PROJECT%20/ds_proj_meth/CRISP_DM_Housing_Prices.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Проект: Домашнее задание по теме «Основы нейронных сетей»  
**Автор:** Кондратьев Степан  
**Дата:** апрель 2025  
**Цель:** Научиться обучать простейшую нейросетевую модель (эквивалентную линейной регрессии) с помощью метода градиентного спуска в PyTorch на примере предсказания стоимости жилья.

# Постановка задачи

### **Задача:**  
Реализовать обучение нейронной сети из одного нейрона (эквивалентной линейной регрессии) для предсказания стоимости квартир на датасете Boston House Prices или California Housing Prices с использованием PyTorch.  

### **Шаги:**

**Понимание данных:**  
- Загрузить выбранный датасет (например, California Housing Prices).  
- Изучить структуру данных: входные признаки (фичи) и целевую переменную (цена жилья).  
- Провести первичный анализ: статистики, распределения, наличие пропусков.  

**Подготовка данных:**  
- Нормализовать или стандартизировать признаки для улучшения сходимости градиентного спуска.  
- Разделить данные на обучающую (`train`) и тестовую (`test`) выборки.  

**Моделирование:**  
- Создать модель с одним линейным слоем (`nn.Linear`) с помощью `torch.nn.Sequential`.  
- Определить функцию потерь (MSE) и оптимизатор (SGD или Adam).  
- Реализовать цикл обучения с градиентным спуском на `train`-данных.  

**Оценка модели:**  
- Проверить качество модели на тестовой выборке, вычислив MSE или другую метрику регрессии.  

### **Результат:**  

Обученная модель с одним нейроном, предсказывающая цену жилья, и оценка её качества на тестовых данных.


# Понимание данных (загрузка, описание, анализ)

In [ ]:
# Импорт библиотек
import polars as pl  # Для загрузки и обработки данных (альтернатива pandas)
import torch  # Основной фреймворк для работы с нейронными сетями
from torch import nn  # Для создания нейросетевых слоев
from sklearn.model_selection import train_test_split  # Для разделения данных на train/test
from sklearn.preprocessing import StandardScaler  # Для стандартизации данных
from sklearn.datasets import fetch_california_housing  # Для загрузки датасета California Housing
import numpy as np  # Импорт библиотеки NumPy для работы с массивами

Загрузка и первичный анализ данных

In [ ]:
# Настройка отображения
pl.Config.set_tbl_cols(-1)  # Показать все столбцы
pl.Config.set_tbl_width_chars(120)  # Ширина таблицы

In [ ]:
# Загрузка данных
california = fetch_california_housing(as_frame=True)
df = pl.DataFrame(california.data)
df = df.with_columns(pl.Series("MedHouseVal", california.target))  # Добавляем целевую переменную

In [ ]:
# Вывод данных для первичного анализа
print("Данные обучающей выборки:")
print(df)

```
Данные обучающей выборки:
shape: (20_640, 9)
┌────────┬──────────┬──────────┬───────────┬────────────┬──────────┬──────────┬───────────┬─────────────┐
│ MedInc ┆ HouseAge ┆ AveRooms ┆ AveBedrms ┆ Population ┆ AveOccup ┆ Latitude ┆ Longitude ┆ MedHouseVal │
│ ---    ┆ ---      ┆ ---      ┆ ---       ┆ ---        ┆ ---      ┆ ---      ┆ ---       ┆ ---         │
│ f64    ┆ f64      ┆ f64      ┆ f64       ┆ f64        ┆ f64      ┆ f64      ┆ f64       ┆ f64         │
╞════════╪══════════╪══════════╪═══════════╪════════════╪══════════╪══════════╪═══════════╪═════════════╡
│ 8.3252 ┆ 41.0     ┆ 6.984127 ┆ 1.02381   ┆ 322.0      ┆ 2.555556 ┆ 37.88    ┆ -122.23   ┆ 4.526       │
│ 8.3014 ┆ 21.0     ┆ 6.238137 ┆ 0.97188   ┆ 2401.0     ┆ 2.109842 ┆ 37.86    ┆ -122.22   ┆ 3.585       │
│ 7.2574 ┆ 52.0     ┆ 8.288136 ┆ 1.073446  ┆ 496.0      ┆ 2.80226  ┆ 37.85    ┆ -122.24   ┆ 3.521       │
│ …      ┆ …        ┆ …        ┆ …         ┆ …          ┆ …        ┆ …        ┆ …         ┆ …           │
│ 1.8672 ┆ 18.0     ┆ 5.329513 ┆ 1.17192   ┆ 741.0      ┆ 2.123209 ┆ 39.43    ┆ -121.32   ┆ 0.847       │
│ 2.3886 ┆ 16.0     ┆ 5.254717 ┆ 1.162264  ┆ 1387.0     ┆ 2.616981 ┆ 39.37    ┆ -121.24   ┆ 0.894   
```

Описание переменных

1. **MedInc**: Средний доход населения в районе (в десятках тысяч долларов)
2. **HouseAge**: Средний возраст домов в районе (в годах)
3. **AveRooms**: Среднее количество комнат на одно жилище
4. **AveBedrms**: Среднее количество спален на одно жилище
5. **Population**: Численность населения в районе
6. **AveOccup**: Среднее количество жителей на одно жилище
7. **Latitude**: Географическая широта района
8. **Longitude**: Географическая долгота района
9. **MedHouseVal**: Медианная стоимость дома в районе (в сотнях тысяч долларов) - целевая переменная

In [ ]:
print("Предварительный анализ столбцов:")

for col in df.columns:
    # Уникальные значения для анализа
    unique_vals = df[col].unique()

    # Сортируем значения по длине строкового представления
    sorted_vals = sorted(unique_vals, key=lambda x: (len(str(x)), str(x)))

    # Ограничиваем вывод до 10 значений
    vals_to_show = sorted_vals[:10]
    vals_str = ", ".join([str(v) for v in vals_to_show])
    if len(sorted_vals) > 10:
        vals_str += ", ..."  # Добавляем многоточие если значений больше 10

    print(f"Столбец: '{col}': Уникальные значения: [{vals_str}]")

```
Предварительный анализ столбцов:
Столбец: 'MedInc': Уникальные значения: [0.9, 1.0, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, ...]
Столбец: 'HouseAge': Уникальные значения: [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, ...]
Столбец: 'AveRooms': Уникальные значения: [1.0, 2.0, 2.4, 2.5, 2.6, 2.8, 3.0, 3.1, 3.2, 3.5, ...]
Столбец: 'AveBedrms': Уникальные значения: [0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, ...]
Столбец: 'Population': Уникальные значения: [3.0, 5.0, 6.0, 8.0, 9.0, 11.0, 13.0, 14.0, 15.0, 17.0, ...]
Столбец: 'AveOccup': Уникальные значения: [1.5, 1.6, 1.8, 2.0, 2.2, 2.3, 2.4, 2.5, 2.6, 2.7, ...]
Столбец: 'Latitude': Уникальные значения: [32.6, 32.7, 32.8, 32.9, 33.0, 33.1, 33.2, 33.3, 33.4, 33.5, ...]
Столбец: 'Longitude': Уникальные значения: [-114.6, -115.4, -115.5, -115.6, -115.8, -115.9, -116.0, -116.2, -116.3, -116.4, ...]
Столбец: 'MedHouseVal': Уникальные значения: [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, ...]
```

Промежуточный вывод по типам переменных:

1. **Бинарные переменные**: Отсутствуют (нет переменных, принимающих только два значения).  
2. **Непрерывные переменные**:  
   - Все переменные представлены вещественными числами (`f64`), включая целевую (`MedHouseVal`).  
3. **Ординальные переменные**:  
   - Частично могут быть интерпретированы как ординальные:  
     - `AveRooms` и `AveBedrms` (количество комнат/спален — дискретные значения с естественным порядком).  
     - `HouseAge` (возраст дома — упорядоченная величина).  
4. **Номинальные переменные**: Отсутствуют (все числовые признаки либо непрерывные, либо допускают упорядочивание).  

**Интересные наблюдения**:  
- Географические координаты (`Latitude`, `Longitude`) — непрерывные, но могут потребовать особой обработки (например, кластеризации для учета пространственной корреляции).  
- Переменные `AveRooms` и `AveBedrms` логично преобразовать в целочисленные (округлением), если модель требует дискретных значений.  

**Возможные преобразования**:  
- Для нейросетей стандартизация непрерывных переменных (`MedInc`, `Population`) обязательна.  
- Ординальные переменные (`AveRooms`, `AveBedrms`) можно оставить как есть или нормализовать.  

**Примечание**: Анализ выбросов, пропусков и точных границ — на следующих этапах.



# Подготовка данных (пропуски, нормализация, разделение, преобразование)

In [ ]:
# Проверка явных пропусков (NULL)
null_counts = df.null_count()
print("Количество NULL значений в каждом столбце:")
print(null_counts)

# Проверка неявных пропусков (NaN) для числовых столбцов
print("\nКоличество NaN значений в числовых столбцах:")
for col in df.columns:
    if df[col].dtype in (pl.Float64, pl.Float32):  # Проверяем только float-столбцы
        nan_count = df.filter(pl.col(col).is_nan()).height
        print(f"{col}: {nan_count}")

```
Количество NULL значений в каждом столбце:
shape: (1, 9)
┌────────┬──────────┬──────────┬───────────┬────────────┬──────────┬──────────┬───────────┬─────────────┐
│ MedInc ┆ HouseAge ┆ AveRooms ┆ AveBedrms ┆ Population ┆ AveOccup ┆ Latitude ┆ Longitude ┆ MedHouseVal │
│ ---    ┆ ---      ┆ ---      ┆ ---       ┆ ---        ┆ ---      ┆ ---      ┆ ---       ┆ ---         │
│ u32    ┆ u32      ┆ u32      ┆ u32       ┆ u32        ┆ u32      ┆ u32      ┆ u32       ┆ u32         │
╞════════╪══════════╪══════════╪═══════════╪════════════╪══════════╪══════════╪═══════════╪═════════════╡
│ 0      ┆ 0        ┆ 0        ┆ 0         ┆ 0          ┆ 0        ┆ 0        ┆ 0         ┆ 0           │
└────────┴──────────┴──────────┴───────────┴────────────┴──────────┴──────────┴───────────┴─────────────┘

Количество NaN значений в числовых столбцах:
MedInc: 0
HouseAge: 0
AveRooms: 0
AveBedrms: 0
Population: 0
AveOccup: 0
Latitude: 0
Longitude: 0
MedHouseVal: 0
```

**Анализ пропущенных значений:**

- NULL значений: отсутствуют во всех столбцах  
- NaN значений: отсутствуют во всех числовых столбцах  

**Вывод:** Пропуски в данных отсутствуют.

In [ ]:
# Подготовка признаков (X) и целевой переменной (y)
X = df.drop("MedHouseVal").to_numpy()  # Все колонки, кроме MedHouseVal
y = df["MedHouseVal"].to_numpy().reshape(-1, 1)  # Преобразуем в 2D массив для совместимости

In [ ]:
# Масштабирование данных
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # Нормализуем признаки (среднее=0, std=1)

In [ ]:
# Разделение данных на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42  # 20% данных в тест, фиксированный random_state для воспроизводимости
)

In [ ]:
# Вывод размеров данных
print(f"Размеры данных:")
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

```
Размеры данных:
X_train: (16512, 8), y_train: (16512, 1)
X_test: (4128, 8), y_test: (4128, 1)
```

In [ ]:
# Преобразование данных в тензоры PyTorch
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

In [ ]:
# Создание Dataset и DataLoader для удобной работы с данными
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)  # Объединяем X и y в один Dataset
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [ ]:
# Параметры загрузки данных
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)  # Перемешиваем данные для обучения
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)   # Для теста shuffle не нужен

# Моделирование ()

In [ ]:
# Определение модели
model = nn.Sequential(
    nn.Linear(in_features=8, out_features=1)  # Один полносвязный слой (аналог линейной регрессии)
)

# Печать структуры модели
print("Структура модели:")
print(model)

```
Структура модели:
Sequential(
  (0): Linear(in_features=8, out_features=1, bias=True)
)
```

In [ ]:
# Инициализация функции потерь и оптимизатора
loss_fn = nn.MSELoss()  # MSE для регрессии
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)  # SGD с learning rate = 0.01
# Альтернатива: optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Гиперпараметры
epochs = 100  # Количество эпох
print_interval = 10  # Вывод логов каждые 10 эпох

# История потерь для визуализации
train_loss_history = []

# Обучение модели
for epoch in range(epochs):
    model.train()  # Режим обучения
    running_loss = 0.0

    for batch_X, batch_y in train_loader:
        # Обнуляем градиенты
        optimizer.zero_grad()

        # Forward pass (вычисление предсказаний)
        y_pred = model(batch_X)

        # Вычисление потерь
        loss = loss_fn(y_pred, batch_y)

        # Backward pass (градиенты)
        loss.backward()

        # Шаг оптимизатора
        optimizer.step()

        # Суммируем потери за батч
        running_loss += loss.item()

    # Средние потери за эпоху
    epoch_loss = running_loss / len(train_loader)
    train_loss_history.append(epoch_loss)

    # Вывод метрик
    if (epoch + 1) % print_interval == 0:
        print(f"Эпоха [{epoch + 1}/{epochs}], Loss: {epoch_loss:.4f}")

```
Эпоха [10/100], Loss: 0.5217
Эпоха [20/100], Loss: 0.5198
Эпоха [30/100], Loss: 0.5536
Эпоха [40/100], Loss: 0.5213
Эпоха [50/100], Loss: 0.5228
Эпоха [60/100], Loss: 0.5189
Эпоха [70/100], Loss: 0.5190
Эпоха [80/100], Loss: 0.5229
Эпоха [90/100], Loss: 0.5228
Эпоха [100/100], Loss: 0.5214
```

In [ ]:
# Пример предсказаний (до финальной оценки)
model.eval()  # Режим оценки
with torch.no_grad():
    test_sample = X_test_tensor[:5]
    predictions = model(test_sample)
    print("\nПример предсказаний:")
    print("Истинные значения:", y_test_tensor[:5].flatten().numpy())
    print("Предсказания:", predictions.flatten().numpy())

```
Пример предсказаний:
Истинные значения: [0.477   0.458   5.00001 2.186   2.78   ]
Предсказания: [0.71133256 1.7603954  2.69768    2.842923   2.6002226 ]
```

# Оценка модели

In [ ]:
# Оценка модели на тестовой выборке
model.eval()  # Переводим модель в режим оценки
with torch.no_grad():
    # Получаем предсказания для тестовой выборки
    y_test_pred = model(X_test_tensor)

    # Вычисляем среднюю квадратичную ошибку (MSE)
    test_loss = loss_fn(y_test_pred, y_test_tensor)
    print(f"Тестовая MSE: {test_loss.item():.4f}")

    # Вычисляем RMSE
    rmse = torch.sqrt(test_loss)  # Корень из MSE
    print(f"Тестовая RMSE: {rmse.item():.4f}")

    # Дополнительно: вычислим R^2 (коэффициент детерминации)
    from sklearn.metrics import r2_score

    # Преобразуем тензоры в numpy для вычисления метрики
    y_test_pred_np = y_test_pred.numpy()
    y_test_np = y_test_tensor.numpy()

    r2 = r2_score(y_test_np, y_test_pred_np)
    print(f"Коэффициент детерминации R^2: {r2:.4f}")


```
Тестовая MSE: 0.5555
Тестовая RMSE: 0.7453
Коэффициент детерминации R^2: 0.5761
```

In [ ]:
import matplotlib.pyplot as plt

# Визуализация потерь во время обучения
plt.figure(figsize=(10, 5))
plt.plot(train_loss_history, label='Loss на обучающей выборке')
plt.title('История потерь во время обучения')
plt.xlabel('Эпохи')
plt.ylabel('Потери (MSE)')
plt.legend()
plt.grid()
plt.show()


In [ ]:
import pandas as pd

# Создание DataFrame для истории потерь
loss_df = pd.DataFrame({
    'Эпоха': range(1, epochs + 1),
    'Потери (MSE)': train_loss_history
})

# Вывод таблицы
print(loss_df)

```
    Эпоха  Потери (MSE)
0       1      1.237505
1       2      0.557877
2       3      0.538748
3       4      0.527599
4       5      0.530203
..    ...           ...
95     96      0.520741
96     97      0.522106
97     98      0.523787
98     99      0.520972
99    100      0.521355

[100 rows x 2 columns]
```

In [ ]:
from sklearn.linear_model import LinearRegression  # Импортируем LinearRegression

# Создаем и обучаем модель линейной регрессии
lin_reg_model = LinearRegression()
lin_reg_model.fit(X_train, y_train)  # Обучаем модель на обучающих данных

# Получаем предсказания для тестовой выборки
y_test_pred_lin_reg = lin_reg_model.predict(X_test)

# Вычисляем метрики качества
from sklearn.metrics import mean_squared_error, r2_score

# Вычисляем MSE и RMSE
mse_lin_reg = mean_squared_error(y_test, y_test_pred_lin_reg)
rmse_lin_reg = mse_lin_reg ** 0.5  # Корень из MSE
r2_lin_reg = r2_score(y_test, y_test_pred_lin_reg)

# Выводим результаты
print(f"Линейная регрессия:")
print(f"Тестовая MSE: {mse_lin_reg:.4f}")
print(f"Тестовая RMSE: {rmse_lin_reg:.4f}")
print(f"Коэффициент детерминации R^2: {r2_lin_reg:.4f}")


```
Линейная регрессия:
Тестовая MSE: 0.5559
Тестовая RMSE: 0.7456
Коэффициент детерминации R^2: 0.5758
```

# Выводы по результатам обучения нейронной сети и линейной регрессии  


#### 1. Результаты обучения нейронной сети:
- **Тестовая MSE**: 0.5555
- **Тестовая RMSE**: 0.7453
- **Коэффициент детерминации R²**: 0.5761

#### 2. Результаты линейной регрессии:
- **Тестовая MSE**: 0.5559
- **Тестовая RMSE**: 0.7456
- **Коэффициент детерминации R²**: 0.5758

### Сравнение моделей

- **Ошибки (MSE и RMSE)**:
  - Нейронная сеть показала немного лучшую производительность по сравнению с линейной регрессией. MSE и RMSE нейронной сети (0.5555 и 0.7453 соответственно) ниже, чем у линейной регрессии (0.5559 и 0.7456).
  
- **Коэффициент детерминации R²**:
  - Нейронная сеть также имеет более высокий коэффициент детерминации (0.5761) по сравнению с линейной регрессией (0.5758). Это означает, что нейронная сеть объясняет немного больше вариации в данных.

### Общие выводы
- Оба метода дают сопоставимые результаты, но простая нейронная сеть в данном случае немного лучше справилась с задачей предсказания цен на жилье по сравнению с линейной регрессией.
- Нейронные сети могут предоставить преимущества в более сложных задачах с большим количеством данных и более сложными зависимостями между признаками.

### Рекомендации для дальнейшего улучшения
- Можно попробовать увеличить количество нейронов и слоев в нейронной сети.
- Экспериментировать с различными функциями активации, оптимизаторами и гиперпараметрами.
- Рассмотреть возможность использования регуляризации для предотвращения переобучения.